In [ ]:
# Phase 11 TPU fold farm: up to eight independent d3-recipe jobs, one per chip. GIT_SHA below names the build.
import glob, os, shutil, subprocess, sys, time
# the TPU image lacks the GPU image's imaging packages; pip needs the notebook's internet switch
for _mod, _pip in (('pydicom', 'pydicom'), ('timm', 'timm'), ('sklearn', 'scikit-learn'), ('torchvision', 'torchvision==0.23.0 --no-deps')):
    try:
        __import__(_mod)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *_pip.split()], check=True)
GIT_SHA = 'knee-phase11-tpu-farm2'
SRC = glob.glob('/kaggle/input/**/rsna-knee-src', recursive=True)[0]
COMP_DIR = glob.glob('/kaggle/input/**/rsna-knee-abnormality-detection', recursive=True)[0]
PKG = '/kaggle/working/knee'
os.makedirs(PKG, exist_ok=True)
for fname in os.listdir(SRC):
    if fname.endswith('.py'):
        shutil.copy(os.path.join(SRC, fname), os.path.join(PKG, fname))
sys.path.insert(0, '/kaggle/working')

import numpy as np
import pandas as pd
import torch
# the parent must never touch the TPU: a device is exclusive to one process, and the eight workers
# need all of them. Versions come from package metadata, not from initialising the runtime.
from importlib.metadata import version
print('torch', torch.__version__, '| torch_xla', version('torch_xla'), flush=True)

import inspect
from knee.dataset import PreppedSlotDataset
from knee.model import KneeModel
from knee.reports import confidence_from_agreement, report_group_key
from knee.train import make_folds
# a stale src dataset would silently train a different model
assert hasattr(KneeModel, 'enable_static_shapes'), 'rsna-knee-src predates static shapes'
assert 'random_groups' in inspect.signature(PreppedSlotDataset.__init__).parameters
import knee.tpu_farm, knee.xla_train, knee.reports
assert hasattr(knee.reports, 'soft_bootstrap')
print('src ok |', GIT_SHA, flush=True)


In [ ]:
# GATE -- the corpus prep census, read from the shard manifests rather than
# downloaded: 9.5GB of artifacts do not need to leave Kaggle to be checked.
import numpy as np
import pandas as pd

PREPPED_DIRS = sorted(glob.glob('/kaggle/input/**/prepped', recursive=True))
uid_to_npz = {}
for d in PREPPED_DIRS:
    for f in os.listdir(d):
        if f.endswith('.npz'):
            uid_to_npz[f[:-4]] = os.path.join(d, f)

manifest = pd.concat([pd.read_csv(f) for f in
                      sorted(glob.glob('/kaggle/input/**/prep_manifest_shard*.csv', recursive=True))])
failed = pd.concat([pd.read_csv(f) for f in
                    sorted(glob.glob('/kaggle/input/**/prep_failed_shard*.csv', recursive=True))])
series_meta = pd.concat([pd.read_csv(f) for f in
                         sorted(glob.glob('/kaggle/input/**/prep_series_meta_shard*.csv', recursive=True))])

# One root by symlink, as Phase 5 did: the loader takes a directory and the
# artifacts arrive in four mounted shards. Symlinks cost no space.
NPZ_ROOT = '/kaggle/working/prepped_all'
os.makedirs(NPZ_ROOT, exist_ok=True)
for uid, p in uid_to_npz.items():
    dst = os.path.join(NPZ_ROOT, f'{uid}.npz')
    if not os.path.exists(dst):
        os.symlink(p, dst)

print(f'{len(PREPPED_DIRS)} shards, {len(uid_to_npz)} artifacts, '
      f'{len(manifest)} manifest rows, {len(failed)} failed studies')
assert len(uid_to_npz) == len(manifest), 'artifact count and manifest disagree'
assert len(uid_to_npz) + len(failed) == 4407, 'shards do not cover the corpus'

mm = series_meta['mm_per_px'].unique()
assert len(mm) == 1, f'physical scale is not constant: {mm}'
print(f'PASS: {len(series_meta)} stored series, {series_meta.PixelSpacing.nunique()} distinct '
      f'PixelSpacing values, all at {mm[0]:.5f} mm/px')

from knee.dicom import SLOTS
slot_names = [n for n, _, _ in SLOTS]
fill = manifest.slots_filled.fillna('').str.get_dummies(sep='|')
print(f'\nmean slots/study {manifest.n_slots_filled.mean():.2f} of 6')
for name in slot_names:
    n = int(fill[name].sum()) if name in fill else 0
    print(f'  {name:12s} {n:5d}  {n / len(manifest):6.1%}')
print(f'\ndecode failures: {int(manifest.n_decode_failures.sum())}')
print(f'laterality routes: {dict(manifest.route.value_counts())}')

In [ ]:
# Folds v3 and labels. Folds are regenerated here rather than uploaded, and
# pinned by hash: the experiment discipline requires one frozen fold assignment
# across every Phase 6 run, and a silent drift would make every paired delta
# meaningless without failing anything.
import hashlib

from knee.infer import LABEL_COLUMNS
from knee.metrics import macro_auc, paired_macro_auc_delta, per_label_auc
from knee.train import Timer, evaluate, log_experiment, train_val_split
from torch.utils.data import DataLoader

train_df = pd.read_csv(f'{COMP_DIR}/train.csv')
all_uids = sorted(train_df['StudyInstanceUID'].astype(str))
assert len(all_uids) == 4407

groups = {u: report_group_key(r)
          for u, r in zip(train_df['StudyInstanceUID'].astype(str), train_df['Report'])}
folds = make_folds(all_uids, n_folds=5, seed=0, groups=groups)
FOLDS_PRIMARY_V3_SHA256 = 'f1d6ba7c341f8d2e82241cb4ddd5ca5131b0c6d2033eb6d2e43bcb597cf12868'
assert hashlib.sha256(
    '\n'.join(f'{u},{folds[u]}' for u in sorted(folds)).encode()
).hexdigest() == FOLDS_PRIMARY_V3_SHA256, 'fold assignment drifted from folds_primary_v3.csv'
n_straddle = sum(1 for _, g in pd.DataFrame(
    {'g': [groups[u] for u in all_uids], 'f': [folds[u] for u in all_uids]}
).groupby('g') if g.f.nunique() > 1)
print(f'folds v3 verified; {n_straddle} report groups straddle folds (v2 had 47)')

gold_cols = [c for c in train_df.columns if c not in ('StudyInstanceUID', 'Report')]
gold_df = train_df[train_df[gold_cols].notna().any(axis=1)]
holdout = frozenset(gold_df['StudyInstanceUID'].astype(str))
print(f'{len(holdout)} gold studies held out of both sides of every split')

# Step 3 re-baselines on the same label source Phase 5 trained on; swapping the
# label target is c0, a separate cell, so the two changes stay attributable.
pseudo = pd.read_csv(glob.glob('/kaggle/input/**/pseudo_labels_qwen3_4b.csv', recursive=True)[0])
labels_all = pseudo[['StudyInstanceUID'] + [f'score_{l}' for l in LABEL_COLUMNS]]
labels_all.columns = ['StudyInstanceUID'] + LABEL_COLUMNS

# Two frames, as Phase 5 used: soft scores are the training target, but AUC is
# scored against a binarized one -- roc_auc_score rejects a continuous y_true,
# and the soft target is a confidence, not a label.
labels_eval = labels_all.copy()
labels_eval[LABEL_COLUMNS] = (labels_all[LABEL_COLUMNS].to_numpy(dtype=float) >= 0.5).astype(float)
print(f'{len(labels_all)} pseudo-labelled studies (soft for training, binarized for scoring)')

# c0's alternative target. steven_v4 is the best single public source on the 58
# gold studies (0.8927 vs our 0.8616, paired delta +0.031 [+0.004, +0.058]).
#
# NOT the 5-source rank blend, which scored higher on gold (0.8976): percentile
# ranks are uniform by construction, so every label's target mean is exactly
# 0.500 and all prevalence information is destroyed -- Fracture would be trained
# at 0.5 against a real rate of 0.014. AUC never notices, because AUC reads
# order only. That is exactly why rank-averaging is right for combining
# predictions at submission time and wrong for building a target.
LABEL_SOURCES = {'ours_qwen3_4b': (labels_all, labels_eval)}
steven = glob.glob('/kaggle/input/**/llm_labels_v4_blend.csv', recursive=True)
if steven:
    alt = pd.read_csv(steven[0])
    alt['StudyInstanceUID'] = alt['StudyInstanceUID'].astype(str)
    alt = alt[['StudyInstanceUID'] + LABEL_COLUMNS]
    alt_eval = alt.copy()
    alt_eval[LABEL_COLUMNS] = (alt[LABEL_COLUMNS].to_numpy(dtype=float) >= 0.5).astype(float)
    LABEL_SOURCES['steven_v4'] = (alt, alt_eval)
    rate = alt[LABEL_COLUMNS].mean()
    print(f'steven_v4 loaded: {len(alt)} studies, positive rate '
          f'{rate.min():.3f}-{rate.max():.3f} (ours: '
          f'{labels_all[LABEL_COLUMNS].mean().min():.3f}-'
          f'{labels_all[LABEL_COLUMNS].mean().max():.3f})')
print('label sources available:', list(LABEL_SOURCES))

# Confidence weights, derived here rather than mounted as a CSV so the derivation
# is reproducible where it is used. Four INDEPENDENT labellings vote: ours,
# steven_v4, pilkwang_v2, lixin73. Steven ships full/v2/v4_blend but those are one
# author's successive versions and would triple-count that author's reading;
# yunusgmsoy is excluded for carrying gold values on the 58 held-out rows.
#
# CAVEAT measured on 2026-09-07 and worth re-reading before believing a positive
# result: agreement is not accuracy. Across the 12 labels, disagreement correlates
# +0.40 (n=12, p=0.20) with steven_v4's gold per-label AUC -- the wrong sign.
# Synovitis is the extreme: near-perfect source agreement (1.2% disputed) and the
# worst-labelled of the twelve. confidence_from_agreement normalises to mean 1 PER
# LABEL, which removes exactly that between-label axis, so what is being tested
# here is strictly within-label. The prior is weaker than it looks.
def _conf_source(pattern, use_conf):
    hits = glob.glob(f'/kaggle/input/**/{pattern}', recursive=True)
    if not hits:
        return None
    df = pd.read_csv(hits[0])
    df['StudyInstanceUID'] = df['StudyInstanceUID'].astype(str)
    out = pd.DataFrame({'StudyInstanceUID': df['StudyInstanceUID']})
    for lab in LABEL_COLUMNS:
        if lab not in df.columns:
            return None
        v = pd.to_numeric(df[lab], errors='coerce')
        conf_col = f'{lab}__conf'
        if use_conf and conf_col in df.columns:
            c = pd.to_numeric(df[conf_col], errors='coerce').clip(0, 1).fillna(0.5)
            v = v * c + (1 - v) * (1 - c)
        out[lab] = v
    return out.set_index('StudyInstanceUID')

_votes = [labels_all.set_index('StudyInstanceUID')[LABEL_COLUMNS]]
for _pat, _uc in (('llm_labels_v4_blend.csv', False),
                  ('report_labels_v2.csv', True),
                  ('report_labels_gpt56sol.csv', True)):
    _v = _conf_source(_pat, _uc)
    if _v is not None:
        _votes.append(_v)

# Below three voices, spread is not a meaningful agreement signal -- fail loudly
# rather than train a "weighted" arm that is silently the unweighted one.
assert len(_votes) >= 4, f'only {len(_votes)} label sources mounted; need 4'
CONFIDENCE = confidence_from_agreement(_votes, LABEL_COLUMNS).reset_index()
CONFIDENCE = CONFIDENCE.rename(columns={CONFIDENCE.columns[0]: 'StudyInstanceUID'})
print(f'confidence weights: {len(CONFIDENCE)} studies from {len(_votes)} sources; '
      f'share below 0.5 per label '
      f'{(CONFIDENCE[LABEL_COLUMNS] < 0.5).mean().round(2).to_dict()}')

# The arbiter for c0. Scoring each model against its OWN label source would be
# circular -- that measures how learnable a label set is, not which one trains a
# better model, and each source would be graded by its own marker. The 58 gold
# studies are rubric-graded from images, excluded from every training split, and
# identical for both arms, so they are the only unbiased comparison available
# when the target itself is the variable. n=58 is thin (MCL has 9 positives) and
# the per-label numbers are directional only; the macro is the read.
gold_labels = gold_df[['StudyInstanceUID'] + LABEL_COLUMNS].copy()
gold_labels['StudyInstanceUID'] = gold_labels['StudyInstanceUID'].astype(str)
gold_uids = [u for u in gold_labels['StudyInstanceUID'] if u in uid_to_npz]
print(f'gold arbiter: {len(gold_uids)} of 58 studies have a prepped artifact')

In [ ]:
# Targets, jobs, then one process per chip.
import json
from knee.dicom import SLOTS
from knee.infer import LABEL_COLUMNS
from knee.claude_labels import ATOMS
from knee.reports import blend_sources, soft_bootstrap
from knee.tpu_farm import save_inputs

JOBS = json.loads('[{"name": "c_s0_f1", "fold": 1, "seed": 0, "target": "c"}, {"name": "c_s0_f2", "fold": 2, "seed": 0, "target": "c"}, {"name": "c_s0_f3", "fold": 3, "seed": 0, "target": "c"}, {"name": "c_s0_f4", "fold": 4, "seed": 0, "target": "c"}, {"name": "d_s0_f1", "fold": 1, "seed": 0, "target": "d"}, {"name": "d_s0_f2", "fold": 2, "seed": 0, "target": "d"}, {"name": "d_s0_f3", "fold": 3, "seed": 0, "target": "d"}, {"name": "d_s0_f4", "fold": 4, "seed": 0, "target": "d"}]')
slot_names = [n for n, _, _ in SLOTS]
assert len(_votes) == 4, f'expected four label sources, found {len(_votes)}'

def _blend_of(extra=()):
    b = blend_sources([_votes[1], _votes[0], _votes[3], *extra], LABEL_COLUMNS)
    b.index.name = 'StudyInstanceUID'
    return b

blend3 = _blend_of()
teacher = pd.read_csv(glob.glob('/kaggle/input/**/teacher8_oof.csv', recursive=True)[0])
teacher['StudyInstanceUID'] = teacher['StudyInstanceUID'].astype(str)
teacher = teacher.set_index('StudyInstanceUID')[LABEL_COLUMNS]
assert len(teacher) == 4349
targets = {'control': dict(df=blend3.reset_index(), columns=LABEL_COLUMNS, weights=None)}
mixed_b = soft_bootstrap(blend3, teacher, alpha=0.5, forbidden_uids=holdout)
mixed_b.index.name = 'StudyInstanceUID'
targets['b'] = dict(df=mixed_b.reset_index(), columns=LABEL_COLUMNS, weights=None)

claude_hits = glob.glob('/kaggle/input/**/claude_labels.csv', recursive=True)
if claude_hits and any(j['target'] in ('c', 'd') for j in JOBS):
    claude = pd.read_csv(claude_hits[0])
    claude['StudyInstanceUID'] = claude['StudyInstanceUID'].astype(str)
    claude = claude.set_index('StudyInstanceUID')
    assert len(claude) == 4349 and not set(claude.index) & holdout, 'claude labels: corpus rows only'
    blend4 = _blend_of([claude[LABEL_COLUMNS]])
    mixed_c = soft_bootstrap(blend4, teacher, alpha=0.5, forbidden_uids=holdout)
    mixed_c.index.name = 'StudyInstanceUID'
    targets['c'] = dict(df=mixed_c.reset_index(), columns=LABEL_COLUMNS, weights=None)
    atom_cols = [f'atom:{a}' for a in ATOMS]
    d_df = mixed_c.reset_index().merge(claude[atom_cols].reset_index(), on='StudyInstanceUID', how='left')
    columns = LABEL_COLUMNS + atom_cols
    weights = d_df[['StudyInstanceUID']].copy()
    for col in columns:
        weights[col] = 1.0 if col in LABEL_COLUMNS else 0.3
    targets['d'] = dict(df=d_df, columns=columns, weights=weights)
missing = {j['target'] for j in JOBS} - set(targets)
assert not missing, f'targets not built: {missing}'

_, eval_labels = LABEL_SOURCES['steven_v4']
save_inputs('/kaggle/working/farm_in', jobs=JOBS, targets=targets, eval_labels=eval_labels, gold_labels=gold_labels,
            folds=folds, holdout=holdout, present=set(uid_to_npz), gold_uids=gold_uids, slot_names=slot_names,
            npz_root=NPZ_ROOT, settings={})
print('jobs:', [(j['name'], j['target'], j['fold'], j['seed']) for j in JOBS], flush=True)

env = dict(os.environ, PYTHONPATH='/kaggle/working')
proc = subprocess.Popen([sys.executable, '-m', 'knee.tpu_farm', '/kaggle/working/farm_in', '/kaggle/working/farm_out'],
                        cwd='/kaggle/working', env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                        start_new_session=True)
# last-resort watchdog: a cancelled Kaggle session saves nothing, so end the farm ourselves and keep the outputs
import signal, threading
threading.Timer(80 * 60, lambda: os.killpg(proc.pid, signal.SIGKILL)).start()
t0 = time.time()
for line in proc.stdout:
    if line.startswith('[') or 'Traceback' in line or 'Error' in line:
        print(f'{(time.time() - t0) / 60:6.1f} min {line.rstrip()}', flush=True)
print('farm exit code', proc.wait(), f'after {(time.time() - t0) / 3600:.2f} h', flush=True)


In [ ]:
# What came out.
import glob as _g
if os.path.exists('/kaggle/working/farm_out/launcher.json'):
    print('launcher:', open('/kaggle/working/farm_out/launcher.json').read())
for p in sorted(_g.glob('/kaggle/working/farm_out/summary_*.json')):
    print(open(p).read())
for p in sorted(_g.glob('/kaggle/working/farm_out/error_*.txt')):
    print('ERROR', p, open(p).read()[-1500:])
